# 3. Ekstraksi fitur

Fitur dihitung ulang dari string URL memakai `app/feature_extraction.py`. Modul yang sama dipakai
oleh website, jadi tidak ada perbedaan cara hitung antara data latih dan input pengguna.

Sebagai pengecekan, nilai hasil hitungan sendiri dibandingkan dengan kolom fitur yang sudah ada
di CSV asli.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "app").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from app.feature_extraction import FEATURE_NAMES, extract_feature_dataframe

PROCESSED = ROOT / "dataset" / "processed"
bersih = pd.read_csv(PROCESSED / "data_bersih.csv")

X = extract_feature_dataframe(bersih["URL"])
print(X.head().to_string())
print("\nJumlah fitur:", len(FEATURE_NAMES))

   url_length  has_ip_address  dot_count  https_flag  url_entropy  token_count  subdomain_count  query_param_count  tld_length  path_length  has_hyphen_in_domain  number_of_digits  tld_popularity  suspicious_file_extension  domain_name_length  percentage_numeric_chars
0          47               0          2           1     4.250669            6                1                  1           3            1                     1                 0               1                          0                  12                       0.0
1          40               0          2           1     4.196439            6                1                  1           3            1                     0                 2               1                          0                  12                       5.0
2          23               0          1           0     3.936180            5                0                  1           2            1                     0                 0              

In [2]:
persentase = {}
for kolom in FEATURE_NAMES:
    hasil = X[kolom].to_numpy()
    asli = bersih[kolom].to_numpy()
    if X[kolom].dtype.kind == "f":
        cocok = abs(hasil - asli) < 1e-6
    else:
        cocok = hasil == asli
    persentase[kolom] = round(float(cocok.mean()) * 100, 3)

print(pd.Series(persentase).sort_values().to_string())

suspicious_file_extension     99.932
has_ip_address                99.978
domain_name_length            99.992
path_length                   99.999
url_entropy                  100.000
url_length                   100.000
dot_count                    100.000
https_flag                   100.000
query_param_count            100.000
subdomain_count              100.000
token_count                  100.000
tld_length                   100.000
number_of_digits             100.000
has_hyphen_in_domain         100.000
tld_popularity               100.000
percentage_numeric_chars     100.000


In [3]:
fitur = X.copy()
fitur["ClassLabel"] = bersih["ClassLabel"]
fitur.to_csv(PROCESSED / "fitur.csv", index=False)
print("Tersimpan di", PROCESSED / "fitur.csv")

Tersimpan di C:\skripsi\phishing-url-detector\dataset\processed\fitur.csv
